# Data Exploration 
Processing football players data for Europe's top 5 leagues in seasons 2017 - 2024

### 1. Import Libraries

In [19]:
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import pandas as pd
import numpy as np
import os

### 2. Import Data

In [20]:
df_original = []
for file in os.listdir('./datasets'):
    if file.startswith('cleaned'):
        if len(df_original) == 0:
            df_original = pd.read_csv(f"./datasets/{file}")
        else:
            df = pd.read_csv(f"./datasets/{file}")
            df_original = pd.concat([df_original, df], axis=0)

print(f'The complete dataset size is {df_original.shape}')

The complete dataset size is (18243, 65)


In [21]:
df_original.head()

,rk,player,nation,pos,squad,comp,age,born,Matches Played,Avg Mins per Match,...,Total Shots,% Shots on target,Shots p 90,Goals per shot,Goals per shot on target,% Aerial Duels won,Shot creating actions p 90,Goal creating actions p 90,Crosses Stopped,season
0,1,Patrick van Aanholt,Netherlands,DF,Crystal Palace,Premier League,26,1990,28,2184,...,33,33.3,0.45,0.15,0.45,54.5,1.90,0.16,0.0,2017-2018
1,2,Rolando Aarons,England,MF,Newcastle Utd,Premier League,21,1995,4,139,...,2,0.0,0.00,0.00,0.00,25.0,0.65,0.00,0.0,2017-2018
2,3,Rolando Aarons,England,MF,Hellas Verona,Serie A,21,1995,11,517,...,3,0.0,0.00,0.00,0.00,37.5,0.87,0.00,0.0,2017-2018
3,4,Ignazio Abate,Italy,DF,Milan,Serie A,30,1986,17,1057,...,4,50.0,0.17,0.25,0.50,55.6,2.30,0.34,0.0,2017-2018
4,5,Aymen Abdennour,Tunisia,DF,Marseille,Ligue 1,27,1989,8,499,...,2,50.0,0.18,0.00,0.00,50.0,0.36,0.00,0.0,2017-2018


### 3. Explore the Data

In [22]:
df_original.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18243 entries, 0 to 2622
Data columns (total 65 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   rk                            18243 non-null  int64  
 1   player                        18243 non-null  object 
 2   nation                        18243 non-null  object 
 3   pos                           18243 non-null  object 
 4   squad                         18243 non-null  object 
 5   comp                          18243 non-null  object 
 6   age                           18243 non-null  int64  
 7   born                          18243 non-null  int64  
 8   Matches Played                18243 non-null  int64  
 9   Avg Mins per Match            18243 non-null  int64  
 10  Goals                         18243 non-null  int64  
 11  Assists                       18243 non-null  int64  
 12  Goals & Assists               18243 non-null  int64  
 13  Non Pen

#### 3.1 Check Missing Values

In [23]:
# Check if there are missing values

df_original.isna().sum()

rk                            0
player                        0
nation                        0
pos                           0
squad                         0
                             ..
% Aerial Duels won            0
Shot creating actions p 90    0
Goal creating actions p 90    0
Crosses Stopped               0
season                        0
Length: 65, dtype: int64

#### 3.2 Check Duplicate Rows

In [24]:
# Check if there are duplicate rows
print(f"Duplicate Rows: {df_original.duplicated().sum()}")

Duplicate Rows: 0


#### 3.3 Ages Histogram for each League and Season

In [129]:
import plotly.graph_objects as go

# ── data setup ─────────────────────────────────────────────────────────────────
leagues = sorted(df_original['comp'].unique().tolist())
seasons = sorted(df_original['season'].unique().tolist())
n_leagues, n_seasons = len(leagues), len(seasons)

# Refined, distinct palette – works great on white background
PALETTE = [
    "#DC2626",  # red
    "#CA8A04",  # amber
    "#EA580C",  # orange
    "#1E4BAA",  # blue
    "#059669",  # emerald
    "#9333EA",  # purple
    "#0891B2",  # cyan
    "#BE185D",  # pink
    "#16A34A",  # green
    "#7C3AED",  # violet
]
league_colors = {lg: PALETTE[i % len(PALETTE)] for i, lg in enumerate(leagues)}

# ── build figure ───────────────────────────────────────────────────────────────
fig = go.Figure()
all_stats = {}

for li, league in enumerate(leagues):
    for si, season in enumerate(seasons):
        sub = df_original[
            (df_original['comp'] == league) &
            (df_original['season'] == season)
        ]['age'].dropna()

        mu  = round(sub.mean(), 1)   if len(sub) else 0
        med = round(sub.median(), 1) if len(sub) else 0
        all_stats[(league, season)] = {"mean": mu, "median": med, "n": len(sub)}

        color = league_colors[league]

        # histogram bars
        fig.add_trace(go.Histogram(
            x=sub,
            name=f"{league} · {season}",
            visible=(li == 0 and si == 0),
            marker=dict(
                color=color,
                opacity=0.80,
                line=dict(color="white", width=1.2),
            ),
            hovertemplate="Age <b>%{x}</b> — Players: <b>%{y}</b><extra></extra>",
            nbinsx=20,
        ))

        # mean reference line (dashed, same color as bars)
        fig.add_trace(go.Scatter(
            x=[mu, mu],
            mode="lines",
            visible=(li == 0 and si == 0),
            line=dict(color=color, dash="dash", width=2),
            showlegend=False,
            hovertemplate=f"Mean age: <b>{mu}</b><extra></extra>",
        ))

# ── visibility helper ──────────────────────────────────────────────────────────
def vis(li, si):
    v = [False] * len(fig.data)
    base = (li * n_seasons + si) * 2
    v[base]     = True
    v[base + 1] = True
    return v

# ── dropdown button factory ────────────────────────────────────────────────────
def make_buttons(fix_dim, fix_idx, varying_items):
    buttons = []
    for idx, item in enumerate(varying_items):
        li = idx      if fix_dim == "league" else fix_idx
        si = idx      if fix_dim == "season" else fix_idx
        st = all_stats[(leagues[li], seasons[si])]
        title = (
            f"<b>Age Distribution</b>  ·  {leagues[li]}  ·  {seasons[si]}<br>"
        )
        buttons.append(dict(
            label=item,
            method="update",
            args=[{"visible": vis(li, si)}, {"title.text": title}],
        ))
    return buttons

league_buttons = make_buttons("league", 0, leagues)
season_buttons = make_buttons("season", 0, seasons)

# ── initial title ──────────────────────────────────────────────────────────────
st0 = all_stats[(leagues[0], seasons[0])]
init_title = (
    f"<b>Age Distribution</b>  ·  {leagues[0]}  ·  {seasons[0]}<br>"
)

# ── shared dropdown style ──────────────────────────────────────────────────────
dropdown_style = dict(
    direction="down",
    showactive=True,
    bgcolor="white",
    bordercolor="#CBD5E1",
    borderwidth=1,
    font=dict(color="#1E293B", size=12, family="Arial"),
)

# ── layout ─────────────────────────────────────────────────────────────────────
fig.update_layout(
    template="plotly_white",
    paper_bgcolor="white",
    plot_bgcolor="#F8FAFC",   # very subtle off-white plot area

    title=dict(
        text=init_title,
        x=0.5,
        xanchor="center",
        font=dict(size=17, color="#0F172A", family="Arial Black"),
    ),

    xaxis=dict(
        title=dict(text="Age (years)", font=dict(size=13, color="#475569")),
        gridcolor="#E2E8F0",
        tickfont=dict(color="#475569", size=11),
        zeroline=False,
        showline=True,
        linecolor="#CBD5E1",
    ),
    yaxis=dict(
        title=dict(text="Player Count", font=dict(size=13, color="#475569")),
        gridcolor="#E2E8F0",
        tickfont=dict(color="#475569", size=11),
        zeroline=False,
        showline=True,
        linecolor="#CBD5E1",
    ),

    updatemenus=[
        dict(
            **dropdown_style,
            buttons=league_buttons,
            pad={"r": 10, "t": 10},
            x=0.75, xanchor="left",
            y=1.14, yanchor="top",
        ),
        dict(
            **dropdown_style,
            buttons=season_buttons,
            pad={"r": 10, "t": 10},
            x=0.895, xanchor="left",
            y=1.14, yanchor="top",
        ),
    ],

    hovermode="x unified",
    hoverlabel=dict(
        bgcolor="white",
        bordercolor="#CBD5E1",
        font_size=12,
        font_color="#0F172A",
        font_family="Arial",
    ),
    bargap=0.08,
    height=580,
    margin=dict(t=150, l=65, r=40, b=65),
    showlegend=False,
)

fig.show()

#### 3.4 

In [68]:
df_original['season'].unique().tolist()

['2017-2018',
 '2018-2019',
 '2019-2020',
 '2020-2021',
 '2021-2022',
 '2022-2023',
 '2023-2024']

### 4. Export the Data

In [25]:
df_original.to_csv('./datasets/full_dataset.csv', index=False)